In [ ]:
import os
import sys
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Configurazione del percorso (Specifico per Colab)
try:
    import google.colab
    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print("Ambiente Colab configurato per l'esecuzione.")
except ImportError:
    print("Ambiente locale rilevato.")

# 2. Gestione del Token di Autenticazione
hf_token = None
try:
    from google.colab import userdata
    print("Acquisizione token da Google Secrets...")
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    print("Lettura token dal file .env locale...")
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    break
    else:
        print("Errore: File .env non trovato.")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Autenticazione Hugging Face completata.")
else:
    print("Attenzione: Token HF non trovato. Possibili errori nel download del modello.")

# 3. Caricamento del Modello (Quantizzazione 4-bit)
model_id = "meta-llama/Llama-4-Scout-17B-16E" 
print(f"Inizializzazione configurazione BitsAndBytes per {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Trasferimento dei pesi del modello in VRAM...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16  # Corretto da torch_dtype
)

print("Caricamento completato con successo. Sistema pronto per l'inferenza.")